# Colab experiments

Reads cached `features_foldA.parquet` and `features_foldB.parquet` from the private Drive folder, runs the
experiment in `CONFIG`, and appends one row to `experiments_colab.csv` in that folder.

**Tune on Fold B only.** Fold A is the untouched report, so do not choose settings by its number.

Mock results go to the mock folder and are labelled `MOCK:`. `scripts/merge_colab_results.py` refuses them in a real run.

## Config

In [ ]:
import os

DEMO = False                                   # True reads parquet files exported from mock data
DRIVE_DIR = "/content/drive/MyDrive/flats_private"   # the private Drive folder
MOCK_DIR = os.environ.get("FLATS_FEATURES_DIR") or f"{DRIVE_DIR}/mock"
DEMO = DEMO or os.environ.get("FLATS_DEMO") == "1"

USE_SRC_MODELS = False   # True calls src.models.run_experiment from an unzipped bundle at BUNDLE_DIR
BUNDLE_DIR = f"{DRIVE_DIR}/colab_bundle"

CONFIG = dict(
    description="LightGBM L1 on log rent",
    model="lightgbm_l1",
    features=None,            # None uses the list stored in the parquet; or give an explicit list
    drop_features=[],         # ablation: names to remove from that list
    log_target=True,
    params=dict(n_estimators=600, learning_rate=0.05, num_leaves=63, min_child_samples=30,
                subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=7),
)

In [ ]:
import csv
import json
import subprocess
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

try:
    import lightgbm as lgb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
    import lightgbm as lgb

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Not on Colab: skipping Drive mount.")

FOLDER = Path(MOCK_DIR if DEMO else DRIVE_DIR)
RESULTS_CSV = FOLDER / "experiments_colab.csv"
FOLDS = {"A": dict(cutoff="2026-05-31", start="2026-06-01", end="2026-07-31"),
         "B": dict(cutoff="2026-03-31", start="2026-04-01", end="2026-05-31")}
# Same order as src/validate.py EXPERIMENT_COLS. *_mae = MEDIAN absolute error, the rank metric.
EXPERIMENT_COLS = ["run_id", "timestamp", "description", "features", "model", "foldA_mae", "foldA_within100",
                   "foldA_r2", "foldB_mae", "bias"]
NEVER_FEATURES = {"rent", "gnaf_id", "building_key", "target_date", "split", "pseudo_cutoff",
                  "advert_car_space", "input_car_space"}

## Scoring

`scored_prediction`, `score` and `round_half` are copied word for word from `src/validate.py`.
If that file changes, copy them again.

In [ ]:
def scored_prediction(df: pd.DataFrame) -> pd.DataFrame:
    """Pick the column matching advert parking, else input parking. Drop only when both are unknown."""
    park = df["advert_car_space"].astype(float).fillna(df["input_car_space"].astype(float))
    out = df[park.notna()].copy()
    p = park[park.notna()]
    out["pred"] = np.where(p > 0, out["rent_with_car_space"], out["rent_without_car_space"])
    return out


def score(df: pd.DataFrame) -> dict:
    """df needs: rent, rent_with_car_space, rent_without_car_space, advert_car_space, input_car_space.

    A blank scored prediction counts as an infinite error, worse than any answered miss.
    """
    s = scored_prediction(df)
    err = (s["pred"] - s["rent"]).astype(float)
    abs_err = err.abs().fillna(np.inf)
    ok = err.notna()
    sse = (err[ok] ** 2).sum()
    sst = ((s.loc[ok, "rent"] - s.loc[ok, "rent"].mean()) ** 2).sum()
    return dict(n=len(s), n_excluded=len(df) - len(s), n_blank=int((~ok).sum()), mae=float(abs_err.median()),
                within100=float((abs_err <= 100).mean()), r2=float(1 - sse / sst) if sst else float("nan"),
                bias=float(err.median()))


def round_half(x) -> np.ndarray:
    return np.round(np.asarray(x, dtype=float) * 2) / 2

## Load and check the cached features

In [ ]:
def load_fold(fold):
    path = FOLDER / f"features_fold{fold}.parquet"
    assert path.exists(), f"Missing {path}. Run scripts/export_features.py and upload the file."
    meta = pq.read_schema(path).metadata or {}
    assert b"flats_source" in meta, f"{path.name} has no metadata. Export it with scripts/export_features.py."
    source = meta[b"flats_source"].decode()
    assert (source == "mock") == DEMO, f"{path.name} holds {source} data but DEMO={DEMO}."
    df = pd.read_parquet(path)
    f = FOLDS[fold]
    train, test = df[df.split == "train"], df[df.split == "test"]
    assert train.target_date.max() <= pd.Timestamp(f["cutoff"]), "train target after the fold cutoff"
    assert test.target_date.between(f["start"], f["end"]).all(), "test row outside the fold window"
    assert (df.target_date > df.pseudo_cutoff).all(), "a row can see its own advert"
    return df, json.loads(meta[b"flats_feature_cols"]), source


data = {k: load_fold(k) for k in FOLDS}
SOURCE = data["A"][2]
stored = data["A"][1]
assert stored == data["B"][1], "folds were exported with different feature lists"
for k, (df, _, _) in data.items():
    print(f"Fold {k}: {df.split.value_counts().to_dict()}, source {SOURCE}")

## Experiment

Each model predicts twice per test row, with `car_space_resolved` set to 1 and then 0, the same two columns a submission
carries. Advert parking on a test row is used only to pick the scored column. There is no early stopping, because
that would tune on the test window.

In [ ]:
def prepare(train, test, cols):
    """Text columns become categories fixed by the train rows; unseen values become NaN."""
    Xtr, Xte = train[cols].copy(), test[cols].copy()
    for c in cols:
        assert not pd.api.types.is_datetime64_any_dtype(Xtr[c]), f"{c} is a date, not a feature"
        if not pd.api.types.is_numeric_dtype(Xtr[c]):
            cats = pd.Index(Xtr[c].dropna().astype(str).unique())
            Xtr[c] = pd.Categorical(Xtr[c].astype(object).where(Xtr[c].notna()), categories=cats)
            Xte[c] = pd.Categorical(Xte[c].astype(object).where(Xte[c].notna()), categories=cats)
    return Xtr, Xte


def run_fold(df, cols, cfg):
    train = df[df.split == "train"]
    test = df[df.split == "test"].reset_index(drop=True)
    Xtr, Xte = prepare(train, test, cols)
    y = np.log(train.rent) if cfg["log_target"] else train.rent
    model = lgb.LGBMRegressor(objective="regression_l1", verbosity=-1, **cfg["params"]).fit(Xtr, y)

    def predict(park):
        X = Xte.assign(car_space_resolved=park) if "car_space_resolved" in cols else Xte
        p = model.predict(X)
        return np.exp(p) if cfg["log_target"] else p

    without = round_half(predict(0.0))
    with_ = np.maximum(round_half(predict(1.0)), without)
    return score(test.assign(rent_with_car_space=with_, rent_without_car_space=without))


def run_notebook_experiment(cfg):
    cols = [c for c in (cfg["features"] or stored) if c not in cfg["drop_features"]]
    bad = NEVER_FEATURES.intersection(cols)
    assert not bad, f"not allowed as features: {bad}"
    return {k: run_fold(data[k][0], cols, cfg) for k in FOLDS}, cols


def run_src_experiment(cfg):
    """Contract assumed from the plan: run_experiment returns a dict with score dicts under 'A' and 'B'."""
    sys.path.insert(0, BUNDLE_DIR)
    from src.models import run_experiment
    out = run_experiment({**cfg, "features_dir": str(FOLDER)})
    assert {"A", "B"} <= set(out), f"run_experiment returned keys {list(out)}; adapt this cell"
    return out, cfg["features"] or stored


results, used = (run_src_experiment if USE_SRC_MODELS else run_notebook_experiment)(CONFIG)
display(pd.DataFrame(results).T)

## Log the result

Appends one row with the same rounding as `log_experiment` in `src/validate.py`. Bias comes from Fold A.

In [ ]:
def append_result(cfg, cols, a, b):
    run_id = uuid.uuid4().hex[:8]
    label = f"MOCK: {cfg['description']}" if SOURCE == "mock" else cfg["description"]
    row = [run_id, datetime.now(timezone.utc).isoformat(timespec="seconds"), label, ",".join(cols), cfg["model"],
           round(a["mae"], 2), round(a["within100"], 4), round(a["r2"], 4), round(b["mae"], 2), round(a["bias"], 2)]
    new = not RESULTS_CSV.exists()
    if not new:
        with open(RESULTS_CSV, newline="", encoding="utf-8") as f:
            assert next(csv.reader(f)) == EXPERIMENT_COLS, f"{RESULTS_CSV} has different columns"
    with open(RESULTS_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if new:
            w.writerow(EXPERIMENT_COLS)
        w.writerow(row)
    return run_id


run_id = append_result(CONFIG, used, results["A"], results["B"])
print(f"run {run_id} appended to {RESULTS_CSV}")
display(pd.read_csv(RESULTS_CSV).tail(5))